# Extracting Features

Once we have extracted scan segments, we treat these as individual observations. Using these observations, we want to construct features from the variable-length time-series data.

We produce datasets that can vary along the following two axes:
1. Simple Statistical Features (mean, range, max, min, stddev) OR Wavelet Features
2. Resolution (we can "chop up" segments of a certain length into smaller pieces)

The segment's lengths can be calculated as the sum of deltas, which are helpfully provided in the csvs folder. The target variable is ID2, which tells us the severity of the overhang on a scale of 0-10.



In [1]:
import os

import numpy as np
import pandas as pd
import scipy.stats as stats
from tqdm import tqdm
import pywt

In [2]:
def simple_statistics(time_series):
    # time_series is a numpy array
    # return the mean, max, min, stddev
    return {
        'mean': np.mean(time_series),
        'max': np.max(time_series),
        'min': np.min(time_series),
        'stddev': np.std(time_series)
    }

# TODO: tune this so it's actually good
# 1. could add entropy of c**2 as an additional feature to capture "spikiness" so we don't just
# average over time
# 2. try db2, db3, db4, db6, db8, sym3, sym6, coif3.

def wavelet_features(signal, wavelet='db4', level=5, include_entropy=False):
    # db4 means that there's 4 vanishing moments, only picks up on oscillations beyond cubic
    coeffs = pywt.wavedec(signal, wavelet, level=min(level, pywt.dwt_max_level(len(signal), wavelet)))
    # Energy in each sub-band (approximation + details)
    energies = [np.sum(c**2) / len(c) for c in coeffs]
    total = sum(energies)
    # Issue with nan/division by 0 in some files
    if total == 0:
        return {f'wavelet_level_{i}_energy_ratio': 0 for i in range(len(coeffs))}
    result = {f'wavelet_level_{i}_energy_ratio': e/total for i, e in enumerate(energies)}
    if include_entropy:
        for i, c in enumerate(coeffs):
            result[f'wavelet_level_{i}_entropy'] = stats.entropy(c**2)
    return result


In [ ]:
data_dir = "./csvs3/"
new_data_dir = "./ml_ready_csvs/"
os.makedirs(new_data_dir, exist_ok=True)

files = os.listdir(data_dir)
files = [f for f in files if f.endswith(".csv")]
files.sort(key=lambda x: int(x.split(".")[0][5:]))
do_this_many = 379 # len(files) # NOTE: can be less if you want to test

for wavelet_method in ['db2', 'db3', 'db4', 'sym3']:
    print(f"Starting {wavelet_method}...")
    # parse all files and prepare to put them in a single DataFrame
    df_list = [] # will contain a list of dictionaries
    for f_idx in tqdm(range(do_this_many)):
        file_path = os.path.join(data_dir, files[f_idx])
        df = pd.read_csv(file_path)

        # this represents the severity of the overhang
        assert df['ID1'].nunique() == 1, "Expected only one unique ID1 (bulk?) per file"
        assert df['ID2'].nunique() == 1, "Expected only one unique ID2 (overhang?) per file"
        target = df['ID2'].values[0]

        num_segments = df['scan_number'].nunique()
        for scan_num in range(num_segments):
            segment = df[df['scan_number'] == scan_num]

            if segment.empty:
                continue

            features = {}
            features['file'] = files[f_idx]
            features['scan_number'] = scan_num
            features['mean_laser_current'] = segment['laser_current'].mean()
            features['in_control'] = segment['in_control'].mode().values[0] # most common
            features |= simple_statistics(segment['signal'].values)
            features |= wavelet_features(segment['signal'].values, wavelet=wavelet_method, include_entropy=True)
            df_list.append(features)
    # create a DataFrame from the list of dictionaries
    combined_df = pd.DataFrame(df_list)
    # save the DataFrame to a new CSV file
    combined_df.to_csv(os.path.join(new_data_dir, f"ml_ready_data_wavelet_method_{wavelet_method}.csv"), index=False)

Starting db2...


100%|██████████| 379/379 [00:22<00:00, 16.66it/s]


Starting db3...


100%|██████████| 379/379 [00:22<00:00, 16.67it/s]


Starting db4...


...

100%|██████████| 379/379 [00:22<00:00, 16.59it/s]


Starting sym6...


100%|██████████| 379/379 [00:21<00:00, 17.74it/s]


Starting coif3...


100%|██████████| 379/379 [00:20<00:00, 18.63it/s]
